# 🌱 Bean Leaf Disease Classification — Machine Learning Pipeline

**Problem.** Beans are a key source of protein in East Africa, and two leaf diseases, **angular leaf spot** and **bean rust**, can destroy a large part of a harvest. A farmer should be able to take a photo of a leaf with a phone and get an instant diagnosis.

**Dataset.** *iBean* by the Makerere University AI Lab and the National Crops Resources Research Institute (NaCRRI), Uganda. Field photos of bean leaves taken with smartphones and labelled by agricultural experts, in **3 classes**: `angular_leaf_spot`, `bean_rust`, `healthy` (1,295 images, 500×500 px).

**Notebook outline**
1. Download the data
2. Organise the data into project folders (train / retrain pool / validation / test)
3. Explore the data (data insights)
4. Preprocessing and data augmentation
5. Baseline: vanilla CNN (no optimisation)
6. Optimised model: **pre-trained MobileNetV2** (transfer learning) + data augmentation + dropout + **L2 regularization** + **Adam optimizer** + **early stopping** + learning-rate scheduling + **hyper-parameter tuning** + fine-tuning
7. Evaluation: **Accuracy, Loss, Precision, Recall, F1 score, ROC-AUC**, confusion matrix, classification report
8. Save the model (it becomes the **pre-trained model** that the app's retraining pipeline fine-tunes)
9. Demo samples and retraining data
10. Download the outputs

> **How to run:** open in Google Colab → *Runtime → Change runtime type → T4 GPU* → *Runtime → Run all* (about 15–25 min).
> At the end, download `bean_project_outputs.zip` and this notebook with its outputs (*File → Download → .ipynb*).

In [ ]:
import os, io, json, time, shutil, zipfile, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, log_loss, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')

IMG_SIZE = 224
CLASS_NAMES = ['angular_leaf_spot', 'bean_rust', 'healthy']
DISPLAY = {'angular_leaf_spot': 'Angular leaf spot', 'bean_rust': 'Bean rust', 'healthy': 'Healthy'}
PROJECT = 'bean_project'          # everything the app needs is written here, then zipped
os.makedirs(PROJECT, exist_ok=True)

print('TensorFlow', tf.__version__, '| Keras', getattr(keras, '__version__', '?'))
print('GPU:', tf.config.list_physical_devices('GPU') or 'none found (training will be slower)')

## 1. Download the data
The official files are downloaded from the iBean storage bucket; if that fails the same dataset is fetched from Hugging Face (`AI-Lab-Makerere/beans`).

In [ ]:
RAW = 'raw'
SPLITS = ['train', 'validation', 'test']

def download_from_google_storage():
    for split in SPLITS:
        path = f'{RAW}/{split}.zip'
        urllib.request.urlretrieve(f'https://storage.googleapis.com/ibeans/{split}.zip', path)
        with zipfile.ZipFile(path) as z:
            z.extractall(f'{RAW}/{split}_extracted')

def download_from_hugging_face():
    import subprocess, sys
    try:
        import datasets
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'], check=True)
    from datasets import load_dataset
    ds = load_dataset('AI-Lab-Makerere/beans')
    for split in SPLITS:
        col = 'labels' if 'labels' in ds[split].features else 'label'
        names = ds[split].features[col].names
        for i, ex in enumerate(ds[split]):
            cls = names[ex[col]]
            folder = f'{RAW}/{split}_extracted/{split}/{cls}'
            os.makedirs(folder, exist_ok=True)
            ex['image'].convert('RGB').save(f'{folder}/{cls}_{split}.{i}.jpg', quality=95)

def collect(split):
    # list of (file path, label index) found under raw/<split>_extracted/**/<class name>/
    items = []
    for dirpath, _, files in os.walk(f'{RAW}/{split}_extracted'):
        cls = os.path.basename(dirpath)
        if cls in CLASS_NAMES:
            items += [(os.path.join(dirpath, f), CLASS_NAMES.index(cls))
                      for f in sorted(files) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    return items

os.makedirs(RAW, exist_ok=True)
if not all(collect(s) for s in SPLITS):
    try:
        download_from_google_storage(); source = 'Google Cloud Storage (ibeans)'
    except Exception as e:
        print('Google Storage download failed:', e, '-> using Hugging Face')
        shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW)
        download_from_hugging_face(); source = 'Hugging Face (AI-Lab-Makerere/beans)'
else:
    source = 'already downloaded'

raw = {s: collect(s) for s in SPLITS}
print('Source:', source)
for s in SPLITS:
    print(f'{s:<11} {len(raw[s]):>5} images')
sizes = pd.Series([Image.open(p).size for p, _ in raw['train'][:200]]).value_counts()
print('Original image sizes (first 200 train images):', dict(sizes))

## 2. Organise the data into project folders
* The official **train** split is divided (stratified) into **train (75 %)**, used to build the model here, and a **retrain pool (25 %)** that plays the role of *new data* uploaded later through the app to **retrain** the model.
* The official **validation** split is used for early stopping and hyper-parameter tuning; the **test** split only for the final evaluation.
* Images are resized from 500×500 to the model input size, **224×224**, and saved as JPEG. This keeps the repository small (~20 MB) and the app uses exactly the same files.

In [ ]:
DATA = f'{PROJECT}/data'

def load_image(source, size=IMG_SIZE):
    # identical to src/preprocessing.py :: load_image  (EXIF rotation -> RGB -> resize -> uint8 array)
    img = Image.open(io.BytesIO(source)) if isinstance(source, (bytes, bytearray)) else Image.open(source)
    img = ImageOps.exif_transpose(img).convert('RGB').resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

def export(items, split):
    for path, label in items:
        folder = f'{DATA}/{split}/{CLASS_NAMES[label]}'
        os.makedirs(folder, exist_ok=True)
        name = os.path.splitext(os.path.basename(path))[0] + '.jpg'
        Image.fromarray(load_image(path)).save(f'{folder}/{name}', quality=92)

shutil.rmtree(DATA, ignore_errors=True)
train_items, pool_items = train_test_split(raw['train'], test_size=0.25, random_state=SEED,
                                           stratify=[label for _, label in raw['train']])
export(train_items, 'train')
export(pool_items, 'retrain_pool')
export(raw['validation'], 'validation')
export(raw['test'], 'test')

def load_image_folder(folder):
    # identical to src/preprocessing.py :: load_image_folder
    X, y, paths = [], [], []
    for cls in sorted(os.listdir(folder)):
        if cls not in CLASS_NAMES:
            continue
        for f in sorted(os.listdir(f'{folder}/{cls}')):
            X.append(load_image(f'{folder}/{cls}/{f}')); y.append(CLASS_NAMES.index(cls)); paths.append(f'{folder}/{cls}/{f}')
    return np.stack(X), np.array(y), paths

X_train, y_train, train_paths = load_image_folder(f'{DATA}/train')
X_pool,  y_pool,  _           = load_image_folder(f'{DATA}/retrain_pool')
X_val,   y_val,   _           = load_image_folder(f'{DATA}/validation')
X_test,  y_test,  test_paths  = load_image_folder(f'{DATA}/test')

summary = pd.DataFrame({split: np.bincount(y, minlength=3) for split, y in
                        [('train', y_train), ('retrain_pool', y_pool), ('validation', y_val), ('test', y_test)]},
                       index=CLASS_NAMES)
summary.loc['TOTAL'] = summary.sum()
summary

## 3. Explore the data (data insights)

In [ ]:
ax = summary.drop('TOTAL').plot(kind='bar', figsize=(9, 4), rot=0, colormap='viridis')
ax.set_title('Insight 1 — Number of images per class and split'); ax.set_ylabel('Images')
plt.show()
counts = summary.loc[CLASS_NAMES, 'train']
print(f'The classes are balanced (between {counts.min()} and {counts.max()} training images per class), '
      'so accuracy is a fair metric and no class weighting / re-sampling is needed.')

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(14, 7.5))
for r, cls in enumerate(CLASS_NAMES):
    idx = np.where(y_train == r)[0][:6]
    for c, i in enumerate(idx):
        axes[r, c].imshow(X_train[i]); axes[r, c].axis('off')
    axes[r, 0].set_title(DISPLAY[cls], loc='left', fontsize=12, fontweight='bold')
plt.suptitle('Insight 2 — Example leaves of each class'); plt.tight_layout(); plt.show()
print('Angular leaf spot shows brown angular patches limited by the veins; bean rust shows many small')
print('reddish-brown dots (pustules); healthy leaves are uniformly green. Backgrounds (soil, hands, other')
print('leaves) vary a lot, which is why data augmentation and a pre-trained model help.')

In [ ]:
def colour_stats(X):
    f = X.reshape(len(X), -1, 3).astype(np.float32)
    rgb = f.mean(axis=1)
    return pd.DataFrame({'red': rgb[:, 0], 'green': rgb[:, 1], 'blue': rgb[:, 2],
                         'greenness': rgb[:, 1] / rgb.sum(axis=1),          # share of green in the image
                         'brightness': rgb.mean(axis=1)})

stats = colour_stats(X_train); stats['class'] = [DISPLAY[CLASS_NAMES[i]] for i in y_train]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=stats, x='class', y='greenness', hue='class', legend=False, ax=ax[0], palette='Greens')
ax[0].set_title('Insight 3 — Greenness (share of green in the image)')
sns.kdeplot(data=stats, x='brightness', hue='class', ax=ax[1], fill=True, common_norm=False)
ax[1].set_title('Insight 4 — Image brightness distribution')
plt.tight_layout(); plt.show()
print(stats.groupby('class')[['red', 'green', 'blue', 'greenness', 'brightness']].mean().round(3))

## 4. Preprocessing and data augmentation
| Step | Where | Why |
|---|---|---|
| EXIF rotation, RGB conversion | `load_image()` | phone photos may be rotated / have an alpha channel |
| Resize 500×500 → 224×224 | `load_image()` | input size of MobileNetV2 |
| Train / validation / test split | section 2 | honest evaluation, early stopping |
| **Data augmentation**: random flips, rotation, zoom, contrast | layers inside the model (training only) | leaves can appear at any angle; reduces over-fitting on a small dataset |
| Rescale pixels 0–255 → −1…1 | `Rescaling` layer inside the model | the scaling MobileNetV2 was pre-trained with; being inside the model guarantees the app does the same |

In [ ]:
X_train_f, X_val_f, X_test_f = (a.astype('float32') for a in (X_train, X_val, X_test))
print('Training array :', X_train_f.shape, X_train_f.dtype, f'pixel range {X_train_f.min():.0f}-{X_train_f.max():.0f}')
print('Validation     :', X_val_f.shape, '  Test:', X_test_f.shape)

def make_augmentation():
    return keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
    ], name='augmentation')

aug = make_augmentation()
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(X_train[0]); axes[0].set_title('original'); axes[0].axis('off')
for k in range(1, 6):
    out = aug(X_train_f[:1], training=True)[0].numpy()
    axes[k].imshow(np.clip(out, 0, 255).astype('uint8')); axes[k].set_title(f'augmented {k}'); axes[k].axis('off')
plt.suptitle('Data augmentation: the same leaf seen 5 different ways'); plt.show()

def compute_metrics(model, X, y):
    proba = model.predict(X, verbose=0)
    pred = proba.argmax(axis=1)
    return {'Accuracy': accuracy_score(y, pred),
            'Loss': log_loss(y, np.clip(proba, 1e-7, 1), labels=[0, 1, 2]),
            'Precision': precision_score(y, pred, average='macro', zero_division=0),
            'Recall': recall_score(y, pred, average='macro', zero_division=0),
            'F1 score': f1_score(y, pred, average='macro', zero_division=0),
            'ROC AUC': roc_auc_score(y, proba, multi_class='ovr')}